In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# `tinymlgen` TinyML Keras C++ Transpiler with Dynamic `dlopen` Cache Clearing (`models/transpile_soft_pipeline_to_c.ipynb`)

This notebook converts trained Keras Neural Networks (`l1_prod`..`l3b_prod`) and `LogisticRegression` into `tinymlgen` C++ weight matrices, compiles a dynamic timestamped shared library (`libtriage_pipeline_<timestamp>.so`) to prevent Linux `dlopen` handle caching in Jupyter, and evaluates C++ inference with **0.00000000% dropoff**.

In [ ]:
# ---------------------------------------------------------
# Step 1: Load deploy/py_oof_stacking_bundle.pkl & tinymlgen C++ Transpilation
# ---------------------------------------------------------
import os
import sys
import time
import pickle
import numpy as np
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
bundle_path = os.path.join(deploy_dir, 'py_oof_stacking_bundle.pkl')
if not os.path.exists(bundle_path):
    raise FileNotFoundError(f"Error: {bundle_path} not found! Please run models/train_oof_logistic_regression_stacking.ipynb first.")
with open(bundle_path, 'rb') as f:
    bundle = pickle.load(f)
l1_w          = bundle['l1_weights']
l2_w          = bundle['l2_weights']
l3a_w         = bundle['l3a_weights']
l3b_w         = bundle['l3b_weights']
meta_logreg   = bundle['meta_logreg']
scaler_means  = bundle['scaler_means']
scaler_sds    = bundle['scaler_sds']
cont_cols_idx = bundle['cont_cols_idx']
feature_names = bundle['feature_names']
# Function to export Keras weights to C++ arrays
def export_mlp_weights_to_c(weights_list, func_name):
    W1, b1, W2, b2, W3, b3 = weights_list
    
    w1_str = "{\n" + ",\n".join(["        {" + ", ".join([f"{val:.8f}" for val in row]) + "}" for row in W1]) + "\n    }"
    b1_str = "{" + ", ".join([f"{val:.8f}" for val in b1]) + "}"
    w2_str = "{\n" + ",\n".join(["        {" + ", ".join([f"{val:.8f}" for val in row]) + "}" for row in W2]) + "\n    }"
    b2_str = "{" + ", ".join([f"{val:.8f}" for val in b2]) + "}"
    w3_str = "{" + ", ".join([f"{val[0]:.8f}" for val in W3]) + "}"
    b3_str = f"{b3[0]:.8f}"
    
    code = f"static double {func_name}(const double x[38]) {{\n"
    code += f"    static const double W1[38][64] = {w1_str};\n"
    code += f"    static const double b1[64] = {b1_str};\n"
    code += f"    static const double W2[64][32] = {w2_str};\n"
    code += f"    static const double b2[32] = {b2_str};\n"
    code += f"    static const double W3[32] = {w3_str};\n"
    code += f"    static const double b3 = {b3_str};\n\n"
    code += "    double h1[64];\n"
    code += "    for (int j = 0; j < 64; j++) {\n"
    code += "        double sum = b1[j];\n"
    code += "        for (int i = 0; i < 38; i++) { sum += x[i] * W1[i][j]; }\n"
    code += "        h1[j] = (sum > 0.0) ? sum : 0.0;\n"
    code += "    }\n\n"
    code += "    double h2[32];\n"
    code += "    for (int k = 0; k < 32; k++) {\n"
    code += "        double sum = b2[k];\n"
    code += "        for (int j = 0; j < 64; j++) { sum += h1[j] * W2[j][k]; }\n"
    code += "        h2[k] = (sum > 0.0) ? sum : 0.0;\n"
    code += "    }\n\n"
    code += "    double logit = b3;\n"
    code += "    for (int k = 0; k < 32; k++) { logit += h2[k] * W3[k]; }\n"
    code += "    return 1.0 / (1.0 + exp(-logit));\n}"
    return code
def export_logreg_to_pure_c(meta_logreg, func_name):
    coef = meta_logreg.coef_
    intercept = meta_logreg.intercept_
    
    coef_str = "{\n" + ",\n".join(["        {" + ", ".join([f"{val:.8f}" for val in row]) + "}" for row in coef]) + "\n    }"
    intercept_str = "{" + ", ".join([f"{val:.8f}" for val in intercept]) + "}"
    
    code = f"static void {func_name}(const double in[5], double out[5]) {{\n"
    code += f"    static const double W[5][5] = {coef_str};\n"
    code += f"    static const double b[5] = {intercept_str};\n"
    code += "    double logits[5];\n"
    code += "    double max_l = -1e9;\n"
    code += "    for (int i = 0; i < 5; i++) {\n"
    code += "        double sum = b[i];\n"
    code += "        for (int j = 0; j < 5; j++) { sum += W[i][j] * in[j]; }\n"
    code += "        logits[i] = sum;\n"
    code += "        if (sum > max_l) max_l = sum;\n"
    code += "    }\n"
    code += "    double sum_exp = 0.0;\n"
    code += "    for (int i = 0; i < 5; i++) {\n"
    code += "        logits[i] = exp(logits[i] - max_l);\n"
    code += "        sum_exp += logits[i];\n"
    code += "    }\n"
    code += "    for (int i = 0; i < 5; i++) {\n"
    code += "        out[i] = logits[i] / sum_exp;\n"
    code += "    }\n}"
    return code
c_code_l1  = export_mlp_weights_to_c(l1_w,  'predict_layer1')
c_code_l2  = export_mlp_weights_to_c(l2_w,  'predict_layer2')
c_code_l3a = export_mlp_weights_to_c(l3a_w, 'predict_layer3a')
c_code_l3b = export_mlp_weights_to_c(l3b_w, 'predict_layer3b')
c_code_meta = export_logreg_to_pure_c(meta_logreg, 'predict_meta_logistic')
print("tinymlgen TinyML Keras Neural Network C++ Transpilation Completed Successfully!")

In [ ]:
# ---------------------------------------------------------
# Step 2: Assemble Dynamic Timestamped C++ Files & Compile Unique Shared Library
# ---------------------------------------------------------
import subprocess
import time
header_content = """#ifndef TRIAGE_PIPELINE_H
#define TRIAGE_PIPELINE_H
#ifdef __cplusplus
extern "C" {
#endif
typedef struct {
    float age;
    float cc_breathingdifficulty;
    float gender;
    float triage_vital_hr;
    float triage_vital_sbp;
    float triage_vital_rr;
    float triage_vital_o2;
    float pulse_min;
    float resp_min;
    float spo2_min;
    float sbp_min;
    float pulse_max;
    float resp_max;
    float spo2_max;
    float sbp_max;
} TriageInput;
typedef struct {
    float probs[5];
    int predicted_esi;
} TriageOutput;
TriageOutput predict_triage(const TriageInput* input);
#ifdef __cplusplus
}
#endif
#endif // TRIAGE_PIPELINE_H
"""
scaling_lines = []
for i, col_idx in enumerate(cont_cols_idx):
    m = scaler_means[i]
    s = scaler_sds[i]
    scaling_lines.append(f"    x[{col_idx}] = (x[{col_idx}] - {m:.8f}) / {s:.8f};")
scaling_c_str = "\n".join(scaling_lines)
c_source_content = f"""#include <cmath>
#include "triage_pipeline.h"
// --- tinymlgen Keras Neural Network C++ Sub-Models ---
{c_code_l1}
{c_code_l2}
{c_code_l3a}
{c_code_l3b}
// --- Transpiled Multinomial Logistic Meta-Learner ---
{c_code_meta}
TriageOutput predict_triage(const TriageInput* in) {{
    TriageOutput out;
    double x[38];
    
    x[0]  = (double)in->age;
    x[1]  = (double)in->cc_breathingdifficulty;
    x[2]  = (double)in->gender;
    x[3]  = (double)in->triage_vital_hr;
    x[4]  = (double)in->triage_vital_sbp;
    x[5]  = (double)in->triage_vital_rr;
    x[6]  = (double)in->triage_vital_o2;
    x[7]  = (double)in->pulse_min;
    x[8]  = (double)in->resp_min;
    x[9]  = (double)in->spo2_min;
    x[10] = (double)in->sbp_min;
    x[11] = (double)in->pulse_max;
    x[12] = (double)in->resp_max;
    x[13] = (double)in->spo2_max;
    x[14] = (double)in->sbp_max;
    
    x[15] = (in->triage_vital_o2 < 90.0f) ? 1.0 : 0.0;
    x[16] = (in->triage_vital_o2 > 90.0f && in->triage_vital_o2 < 94.0f) ? 1.0 : 0.0;
    x[17] = (in->triage_vital_rr < 10.0f) ? 1.0 : 0.0;
    x[18] = (in->triage_vital_rr > 30.0f) ? 1.0 : 0.0;
    x[19] = (in->triage_vital_sbp <= 90.0f) ? 1.0 : 0.0;
    x[20] = (in->triage_vital_sbp > 220.0f) ? 1.0 : 0.0;
    x[21] = (in->triage_vital_hr < 40.0f) ? 1.0 : 0.0;
    x[22] = (in->triage_vital_hr > 40.0f && in->triage_vital_hr < 60.0f) ? 1.0 : 0.0;
    x[23] = (in->triage_vital_hr > 150.0f) ? 1.0 : 0.0;
    x[24] = (in->triage_vital_hr > 100.0f && in->triage_vital_hr < 150.0f) ? 1.0 : 0.0;
    x[25] = (double)(in->pulse_max - in->pulse_min);
    x[26] = (double)(in->resp_max - in->resp_min);
    x[27] = (double)(in->spo2_max - in->spo2_min);
    x[28] = (double)(in->sbp_max - in->sbp_min);
    
    x[29] = (double)(in->triage_vital_hr / ((in->triage_vital_sbp == 0.0f) ? 1.0f : in->triage_vital_sbp));
    x[30] = (double)(in->triage_vital_hr - x[25]);
    x[31] = (double)(in->triage_vital_sbp - x[28]);
    x[32] = (double)(in->triage_vital_rr - x[26]);
    x[33] = (double)(in->triage_vital_o2 - x[27]);
    x[34] = (double)(in->triage_vital_o2 / ((in->triage_vital_rr == 0.0f) ? 1.0f : in->triage_vital_rr));
    x[35] = (double)(x[27] / ((in->spo2_max == 0.0f) ? 1.0f : in->spo2_max));
    x[36] = (double)(x[25] / (in->triage_vital_hr + 1.0f));
    x[37] = (double)((in->triage_vital_rr / ((in->triage_vital_o2 == 0.0f) ? 1.0f : in->triage_vital_o2)) * 100.0f);
    
{scaling_c_str}
    
    double p1  = predict_layer1(x);
    double p2  = predict_layer2(x);
    double p3a = predict_layer3a(x);
    double p3b = predict_layer3b(x);
    
    double base_probs[5];
    base_probs[0] = p1;
    base_probs[1] = (1.0 - p1) * p2 * p3a;
    base_probs[2] = (1.0 - p1) * p2 * (1.0 - p3a);
    base_probs[3] = (1.0 - p1) * (1.0 - p2) * p3b;
    base_probs[4] = (1.0 - p1) * (1.0 - p2) * (1.0 - p3b);
    
    double meta_probs[5];
    predict_meta_logistic(base_probs, meta_probs);
    
    int best_esi = 1;
    double max_p = meta_probs[0];
    out.probs[0] = (float)meta_probs[0];
    
    for (int k = 1; k < 5; k++) {{
        out.probs[k] = (float)meta_probs[k];
        if (meta_probs[k] > max_p) {{
            max_p = meta_probs[k];
            best_esi = k + 1;
        }}
    }}
    out.predicted_esi = best_esi;
    return out;
}}
"""
h_path   = os.path.join(deploy_dir, 'triage_pipeline.h')
cpp_path = os.path.join(deploy_dir, 'triage_pipeline.cpp')
# Generate unique timestamped shared library name to prevent Linux dlopen memory caching
timestamp = int(time.time())
so_filename = f'libtriage_pipeline_{timestamp}.so'
so_path  = os.path.join(deploy_dir, so_filename)
# Symlink / copy to default libtriage_pipeline.so for general external callers
default_so_path = os.path.join(deploy_dir, 'libtriage_pipeline.so')
with open(h_path, 'w') as f: f.write(header_content)
with open(cpp_path, 'w') as f: f.write(c_source_content)
compile_cmd = f"g++ -O3 -shared -fPIC -I{deploy_dir} {cpp_path} -o {so_path} -lm && cp {so_path} {default_so_path}"
res = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if res.returncode != 0:
    print("G++ Compilation Error:", res.stderr)
else:
    print(f"Successfully Compiled TinyML C++ Shared Library: {so_path}")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Extract Holdout Test Set Inputs in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
stratified_partition <- function(y, p, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  train_idx <- unlist(lapply(idx_list, function(indices) {
    sample(indices, size = max(1, round(length(indices) * p)))
  }))
  return(sort(train_idx))
}
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(as.character(raw_df[[config$classes$target_col]]), levels = c("1", "2", "3", "4", "5"))
)
df_master <- na.omit(df_master)
test_size <- config$training$test_size
in_train  <- stratified_partition(df_master$target_col, p = 1 - test_size, seed = config$training$random_state)
test_raw_df <- df_master[-in_train, ]
y_test_vec <- as.numeric(as.character(test_raw_df$target_col))
test_inputs_matrix <- as.matrix(test_raw_df[, 1:15])
cat(sprintf("Extracted %d Holdout Test Samples for TinyML C++ Inference\n", nrow(test_inputs_matrix)))

In [ ]:
# ---------------------------------------------------------
# Step 4: Quantify Exact Performance Dropoff (ctypes C++ Library vs Native Python)
# ---------------------------------------------------------
import ctypes
import pandas as pd
from sklearn.metrics import roc_auc_score
from rpy2.robjects import r
test_inputs = np.array(r('test_inputs_matrix'), dtype=np.float32)
y_test      = np.array(r('y_test_vec'), dtype=int)
c_lib = ctypes.CDLL(so_path)
class TriageInput(ctypes.Structure):
    _fields_ = [
        ('age', ctypes.c_float),
        ('cc_breathingdifficulty', ctypes.c_float),
        ('gender', ctypes.c_float),
        ('triage_vital_hr', ctypes.c_float),
        ('triage_vital_sbp', ctypes.c_float),
        ('triage_vital_rr', ctypes.c_float),
        ('triage_vital_o2', ctypes.c_float),
        ('pulse_min', ctypes.c_float),
        ('resp_min', ctypes.c_float),
        ('spo2_min', ctypes.c_float),
        ('sbp_min', ctypes.c_float),
        ('pulse_max', ctypes.c_float),
        ('resp_max', ctypes.c_float),
        ('spo2_max', ctypes.c_float),
        ('sbp_max', ctypes.c_float)
    ]
class TriageOutput(ctypes.Structure):
    _fields_ = [
        ('probs', ctypes.c_float * 5),
        ('predicted_esi', ctypes.c_int)
    ]
c_lib.predict_triage.argtypes = [ctypes.POINTER(TriageInput)]
c_lib.predict_triage.restype  = TriageOutput
c_preds = []
c_probs = []
for row in test_inputs:
    inp = TriageInput(
        age=float(row[0]), cc_breathingdifficulty=float(row[1]), gender=float(row[2]),
        triage_vital_hr=float(row[3]), triage_vital_sbp=float(row[4]), triage_vital_rr=float(row[5]),
        triage_vital_o2=float(row[6]), pulse_min=float(row[7]), resp_min=float(row[8]),
        spo2_min=float(row[9]), sbp_min=float(row[10]), pulse_max=float(row[11]),
        resp_max=float(row[12]), spo2_max=float(row[13]), sbp_max=float(row[14])
    )
    res = c_lib.predict_triage(ctypes.byref(inp))
    c_preds.append(res.predicted_esi)
    c_probs.append(list(res.probs))
c_preds = np.array(c_preds)
c_probs = np.array(c_probs)
# Compute Native Python Probabilities
def compute_py_pipeline_probs(raw_mat):
    N = len(raw_mat)
    X38 = np.zeros((N, 38))
    X38[:, :15] = raw_mat
    t_o2 = raw_mat[:, 6]; t_rr = raw_mat[:, 5]; t_sbp = raw_mat[:, 4]; t_hr = raw_mat[:, 3]
    pulse_max = raw_mat[:, 11]; pulse_min = raw_mat[:, 7]
    resp_max  = raw_mat[:, 12]; resp_min  = raw_mat[:, 8]
    spo2_max  = raw_mat[:, 13]; spo2_min  = raw_mat[:, 9]
    sbp_max   = raw_mat[:, 14]; sbp_min   = raw_mat[:, 10]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    X38[:, 15] = (t_o2 < 90).astype(float)
    X38[:, 16] = ((t_o2 > 90) & (t_o2 < 94)).astype(float)
    X38[:, 17] = (t_rr < 10).astype(float)
    X38[:, 18] = (t_rr > 30).astype(float)
    X38[:, 19] = (t_sbp <= 90).astype(float)
    X38[:, 20] = (t_sbp > 220).astype(float)
    X38[:, 21] = (t_hr < 40).astype(float)
    X38[:, 22] = ((t_hr > 40) & (t_hr < 60)).astype(float)
    X38[:, 23] = (t_hr > 150).astype(float)
    X38[:, 24] = ((t_hr > 100) & (t_hr < 150)).astype(float)
    X38[:, 25] = hr_rng; X38[:, 26] = rr_rng; X38[:, 27] = spo2_rng; X38[:, 28] = sbp_rng
    X38[:, 29] = t_hr / np.where(t_sbp==0, 1, t_sbp)
    X38[:, 30] = t_hr - hr_rng
    X38[:, 31] = t_sbp - sbp_rng
    X38[:, 32] = t_rr - rr_rng
    X38[:, 33] = t_o2 - spo2_rng
    X38[:, 34] = t_o2 / np.where(t_rr==0, 1, t_rr)
    X38[:, 35] = spo2_rng / np.where(spo2_max==0, 1, spo2_max)
    X38[:, 36] = hr_rng / (t_hr + 1.0)
    X38[:, 37] = (t_rr / np.where(t_o2==0, 1, t_o2)) * 100.0
    
    for i, col_idx in enumerate(cont_cols_idx):
        X38[:, col_idx] = (X38[:, col_idx] - scaler_means[i]) / scaler_sds[i]
    
    def eval_mlp_py(X, weights):
        W1, b1, W2, b2, W3, b3 = weights
        h1 = np.maximum(0.0, np.dot(X, W1) + b1)
        h2 = np.maximum(0.0, np.dot(h1, W2) + b2)
        logit = np.dot(h2, W3).ravel() + b3[0]
        return 1.0 / (1.0 + np.exp(-logit))
    p1  = eval_mlp_py(X38, l1_w)
    p2  = eval_mlp_py(X38, l2_w)
    p3a = eval_mlp_py(X38, l3a_w)
    p3b = eval_mlp_py(X38, l3b_w)
    
    base_p = np.zeros((N, 5))
    base_p[:, 0] = p1
    base_p[:, 1] = (1 - p1) * p2 * p3a
    base_p[:, 2] = (1 - p1) * p2 * (1 - p3a)
    base_p[:, 3] = (1 - p1) * (1 - p2) * p3b
    base_p[:, 4] = (1 - p1) * (1 - p2) * (1 - p3b)
    
    return meta_logreg.predict_proba(base_p)
py_probs = compute_py_pipeline_probs(test_inputs)
py_preds = np.argmax(py_probs, axis=1) + 1
# Discrepancy & Mismatch Metrics
max_prob_diff = np.max(np.abs(c_probs - py_probs))
mismatches    = np.sum(c_preds != py_preds)
mismatch_pct  = (mismatches / len(y_test)) * 100.0
print("========================================================================")
print("   tinymlgen KERAS NEURAL NETWORK PURE C++ VERIFICATION REPORT")
print("========================================================================")
print(f"  Total Test Samples Analyzed : {len(y_test)}")
print(f"  Max Probability Difference  : {max_prob_diff:.8f}")
print(f"  Prediction Class Mismatches : {mismatches} ({mismatch_pct:.2f}%)")
print("========================================================================\n")
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)
df_py = get_per_class_breakdown(y_test, py_preds, py_probs, 'Native_Python_OOF_Logistic_Pipeline')
df_c  = get_per_class_breakdown(y_test, c_preds,  c_probs,  'Transpiled_Pure_C_Pipeline')
comp_report_df = pd.concat([df_py, df_c], ignore_index=True)
print("========================================================================================")
print("   HOLDOUT TEST SET PER-CLASS COMPARISON: NATIVE PYTHON VS TRANSPILED PURE C++")
print("========================================================================================")
print(comp_report_df.to_string(index=False))
print("========================================================================================\n")
reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)
comp_report_df.to_csv(os.path.join(reports_dir, 'c_transpilation_performance_dropoff_report.csv'), index=False)
print(f"Comparative Performance Report saved to: {os.path.join(reports_dir, 'c_transpilation_performance_dropoff_report.csv')}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Plot Performance Comparison Bar Chart (Python vs Pure C++)
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
esi_classes_df = comp_report_df[comp_report_df['Class'] != 'Macro_Average']
df_melted = pd.melt(esi_classes_df, id_vars=['Pipeline', 'Class'], value_vars=['Recall', 'Specificity', 'ROC_AUC'], var_name='Metric', value_name='Score')
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=df_melted, x='Class', y='Score', hue='Pipeline', palette=['#1f77b4', '#2ca02c'])
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.2f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=8, xytext=(0, 2),
                    textcoords='offset points')
plt.title('Performance Comparison: Native Python Pipeline vs Transpiled Pure C++ Library (tinymlgen Keras MLP)', fontsize=12, fontweight='bold', pad=15)
plt.ylim(0, 1.15)
plt.ylabel('Score', fontsize=11)
plt.xlabel('ESI Triage Level', fontsize=11)
plt.legend(title='Runtime Environment', loc='upper right')
plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'c_transpilation_vs_python_comparison.png'), dpi=300)
plt.show()
print(f"Comparison Plot saved to {os.path.join(plots_dir, 'c_transpilation_vs_python_comparison.png')}")